# QUERY 5
Cantidad de transacciones del período [2022-09-01, 2022-09-05] con formato de pago
"Wire" o "ACH" cuyo monto convertido a USD sea menor a 1

In [7]:
import numpy as np
import pandas as pd
import requests

RUTA_DE_DATASETS = "../../datasets"
NOMBRE_ARCHIVO = "transacciones_sample.csv"
SOLUCION = "q5_solucion.csv"

transacciones = pd.read_csv(f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO}")
print(f"Total filas cargadas: {len(transacciones)}")
transacciones.head()


Total filas cargadas: 1400000


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/03 03:24,136474,80D515040,135933,80E5487B0,930.16,Canadian Dollar,930.16,Canadian Dollar,Cheque,0
1,2022/09/09 10:05,23726,802E935D0,10876,805F5F7F0,168.73,Euro,168.73,Euro,Credit Card,0
2,2022/09/09 17:24,125980,80DE66D10,119927,819480830,133.83,US Dollar,133.83,US Dollar,Credit Card,0
3,2022/09/04 10:50,1231,800EB82B0,22481,805819E10,2574.04,US Dollar,2574.04,US Dollar,Cheque,0
4,2022/09/05 01:23,1835,8008903D0,137519,814C88EE0,1499.27,Euro,1499.27,Euro,Cheque,0


In [8]:
# Filtro por período [2022-09-01, 2022-09-05]
# "2022/09/01" <= Timestamp <= "2022/09/05" (inclusive)
INICIO = "2022/09/01"
FIN    = "2022/09/06"  # "2022/09/06" para incluir todo el día 5/9

filtro_periodo = transacciones[
    (transacciones["Timestamp"] >= INICIO) &
    (transacciones["Timestamp"] < FIN)
]
print(f"Transacciones en el período: {len(filtro_periodo)}")
filtro_periodo.head()

Transacciones en el período: 763259


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/03 03:24,136474,80D515040,135933,80E5487B0,930.16,Canadian Dollar,930.16,Canadian Dollar,Cheque,0
3,2022/09/04 10:50,1231,800EB82B0,22481,805819E10,2574.04,US Dollar,2574.04,US Dollar,Cheque,0
4,2022/09/05 01:23,1835,8008903D0,137519,814C88EE0,1499.27,Euro,1499.27,Euro,Cheque,0
5,2022/09/02 20:22,70,10042B660,1120,801264710,37.82,US Dollar,37.82,US Dollar,Credit Card,0
7,2022/09/01 00:01,21048,8127856E0,21048,8127856E0,12.23,Euro,12.23,Euro,Reinvestment,0


In [9]:
# Filtro por formato de pago: Wire o ACH
FORMATOS = ["Wire", "ACH"]

filtro_formato = filtro_periodo[filtro_periodo["Payment Format"].isin(FORMATOS)]
print(f"Transacciones Wire/ACH en el período: {len(filtro_formato)}")
filtro_formato.head()


Transacciones Wire/ACH en el período: 105347


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
15,2022/09/03 01:33,136136,80D4BC550,235676,80D4D30F0,70.38,Canadian Dollar,70.38,Canadian Dollar,Wire,0
19,2022/09/01 00:13,123651,8096A61D0,119699,8096C6BB0,6524642.17,Rupee,6524642.17,Rupee,ACH,0
29,2022/09/01 05:14,354400,813CE00F0,253379,813CE0B20,1552.08,Swiss Franc,1552.08,Swiss Franc,ACH,0
36,2022/09/04 10:56,153484,8183EE710,147509,8182A13E0,758.95,US Dollar,758.95,US Dollar,ACH,0
38,2022/09/03 04:27,115332,8076E8320,16298,8156D5470,8.65,Euro,8.65,Euro,ACH,0


In [10]:
# Descarga de cotizaciones desde API Frankfurter (base EUR)
# y forward-fill para días sin mercado (fines de semana / feriados)
from datetime import date, timedelta

url = "https://api.frankfurter.app/2022-09-01..2022-09-05"
resp = requests.get(url, timeout=10)
resp.raise_for_status()
cotizaciones_raw = resp.json()["rates"]   # solo días hábiles
print("Fechas con cotización de la API:", list(cotizaciones_raw.keys()))

# Expandir el rango completo usando la última cotización conocida
cotizaciones = {}
last_rates = None
dia = date(2022, 9, 1)
fin = date(2022, 9, 5)
while dia <= fin:
    key = dia.strftime("%Y-%m-%d")
    if key in cotizaciones_raw:
        last_rates = cotizaciones_raw[key]
    if last_rates is not None:
        cotizaciones[key] = last_rates
    dia += timedelta(days=1)

print("Fechas disponibles tras forward-fill:", list(cotizaciones.keys()))


Fechas con cotización de la API: ['2022-09-01', '2022-09-02', '2022-09-05']
Fechas disponibles tras forward-fill: ['2022-09-01', '2022-09-02', '2022-09-03', '2022-09-04', '2022-09-05']


In [11]:
# Conversión de monto recibido a USD (misma lógica que el worker distribuido)
CURRENCY_MAP = {
    "US Dollar":         "USD",
    "Euro":              "EUR",
    "UK Pound":          "GBP",
    "Yen":               "JPY",
    "Australian Dollar": "AUD",
    "Bitcoin":           "BTC",
    "Brazil Real":       "BRL",
    "Canadian Dollar":   "CAD",
    "Mexican Peso":      "MXN",
    "Ruble":             "RUB",
    "Rupee":             "INR",
    "Saudi Riyal":       "SAR",
    "Shekel":            "ILS",
    "Swiss Franc":       "CHF",
    "Yuan":              "CNY",
}

def convertir_a_usd(row):
    iso = CURRENCY_MAP.get(row["Receiving Currency"])
    if not iso:
        return None
    if iso == "USD":
        return float(row["Amount Received"])

    fecha = row["Timestamp"].split(" ")[0].replace("/", "-")
    rates = cotizaciones.get(fecha)  # ya tiene forward-fill para fines de semana
    if not rates:
        return None

    rate_usd = rates.get("USD")
    if not rate_usd:
        return None

    monto = float(row["Amount Received"])
    if iso == "EUR":
        return monto * rate_usd

    # Triangulación: origen → EUR → USD
    rate_origen = rates.get(iso)
    if not rate_origen:
        return None
    return (monto / rate_origen) * rate_usd

filtro_formato = filtro_formato.copy()
filtro_formato["amount_usd"] = filtro_formato.apply(convertir_a_usd, axis=1)

# Descartamos filas sin cotización (divisa no mapeada) y filtramos monto < 1 USD
menores_1_usd = filtro_formato.dropna(subset=["amount_usd"])
menores_1_usd = menores_1_usd[menores_1_usd["amount_usd"] < 1.0]
print(f"Transacciones < 1 USD: {len(menores_1_usd)}")
menores_1_usd[["Timestamp", "Receiving Currency", "Amount Received", "amount_usd"]].head()


Transacciones < 1 USD: 1512


,Timestamp,Receiving Currency,Amount Received,amount_usd
335,2022/09/01 00:09,Yuan,0.01,0.001449
589,2022/09/03 20:13,Yuan,0.66,0.095542
1352,2022/09/01 04:20,Rupee,12.70,0.159572
2945,2022/09/01 18:07,US Dollar,0.33,0.330000
3023,2022/09/05 10:19,Rupee,53.73,0.672700


In [12]:
# Resultado final: cantidad de transacciones
count = len(menores_1_usd)
resultado = pd.DataFrame({"count": [count]})
print(f"Q5 resultado: {count} transacciones")

resultado.to_csv(SOLUCION, index=False)
print(f"Guardado en {SOLUCION}")


Q5 resultado: 1512 transacciones
Guardado en q5_solucion.csv
